# Step 4 — LightGBM ranker

Ranks each search's list with LightGBM (LambdaRank) on the 28 step-2 features. No step-3 scores as inputs and no blending.

**Setup.** Same as steps 1–3: 70/15/15 split by search (seed 4222), full lists, NDCG@10 with gains 0 / 1 / 3 / 7, and ties broken at random. Validation picks the settings, and test is scored once, in Section 7.

**Features.** Step 2's Sections 1, 2, 3 and 5 (28 features). Two groups are left out:
- Section 4: one value per search, so it can't reorder a list.
- Section 6 (#29–36, same-day label rates): they count outcomes of other searches on 17 Oct, some later than the search being scored, and there is no time of day to keep only the earlier ones. They are not loaded anywhere in this notebook.

**Where to find the results.**
- Tuning and ablations: end of Sections 2 and 3
- Train and validation against step 3: Section 6
- Test and paired tests: Section 7
- Segments: Section 8

| Section | Content | Output (`lightgbm/`) |
|---|---|---|
| 0 | Setup | — |
| 1 | Load and check the features | — |
| 2 | Tuning on 25% of training searches | `runs.csv` |
| 3 | Ablations by step-2 section | `runs.csv` |
| 4 | Final models on all training searches | `model_*.txt`, `score_*.parquet` |
| 5 | Feature importance | `importance.csv` |
| 6 | Train and validation results | `results.csv` |
| 7 | Test results | `test.csv`, `paired_tests.csv` |
| 8 | Segments (validation) | `segments.csv` |

Score files hold `sid`, `wid`, `score`, one row per shown product in exposures order (same as step 3).

## Section 0 · Setup

- On Colab this mounts Drive and uses the project folders. Locally it reads everything from the notebook's folder.
- `ndcg_lists`: NDCG@10 of each search (same metric as steps 2–3). `ndcg_at_k`: its mean.
- `paired_test`: gap between two models on the same searches, with a 95% interval and a paired t-test.
- `RUN_TEST = True` because this is the final model, so Section 7 scores test.

In [1]:
import os, sys, gc, time, psutil
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import stats

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q lightgbm

import lightgbm as lgb

ROOT = "/content/drive/MyDrive/BT4222_Group10_JDSearch" if IN_COLAB else "."
OUT  = f"{ROOT}/processed" if IN_COLAB else ROOT   # step-1 tables
FEAT = f"{ROOT}/features" if IN_COLAB else ROOT    # step-2 features
LGB  = f"{ROOT}/lightgbm"                          # this notebook's output
os.makedirs(LGB, exist_ok=True)

SEED = 4222
SPLITS = {"train": 0, "valid": 1, "test": 2}
GAIN = np.array([0.0, 1.0, 3.0, 7.0])      # gain per label: 2^label - 1
TUNE_SHARE = 0.25                          # share of training searches used in Sections 2-3
RUN_TEST = True                            # final model, so Section 7 scores test

def ram(tag=""):
    print(f"[{tag}] RAM in use: {psutil.Process().memory_info().rss / 1e9:.1f} GB")

def ndcg_lists(sid, label, score, k=10, seed=0):
    """NDCG@k of each search. Rank by score (high first, NaN last, ties at random); searches with no interaction are dropped."""
    d = pd.DataFrame({"sid": sid, "label": label, "score": score})
    d["tie"] = np.random.default_rng(seed).random(len(d))
    d = d.sort_values(["sid", "score", "tie"], ascending=[True, False, True], na_position="last")
    pos = d.groupby("sid").cumcount().to_numpy()                   # 0 = top
    dcg = pd.Series(np.where(pos < k, GAIN[d.label.to_numpy()] / np.log2(pos + 2), 0.0)).groupby(d.sid.to_numpy()).sum()
    best = d.sort_values(["sid", "label"], ascending=[True, False]) # ideal order
    bpos = best.groupby("sid").cumcount().to_numpy()
    idcg = pd.Series(np.where(bpos < k, GAIN[best.label.to_numpy()] / np.log2(bpos + 2), 0.0)).groupby(best.sid.to_numpy()).sum()
    ok = idcg > 0
    return dcg[ok] / idcg[ok]

def ndcg_at_k(sid, label, score, k=10, seed=0):
    """Mean NDCG@k over searches (same metric as steps 2-3)."""
    return float(ndcg_lists(sid, label, score, k, seed).mean())

def paired_test(a, b):
    """a - b on the same searches (two outputs of ndcg_lists): mean gap, 95% interval, p-value, share of searches better / worse."""
    d = a - b
    half = 1.96 * d.std() / np.sqrt(len(d))
    return {"difference": d.mean(), "95% low": d.mean() - half, "95% high": d.mean() + half,
            "p-value": stats.ttest_rel(a, b).pvalue, "share better": (d > 0).mean(), "share worse": (d < 0).mean()}

print("LightGBM", lgb.__version__, "| pandas", pd.__version__, "| threads", os.cpu_count(), "| Colab", IN_COLAB)
ram("setup")

LightGBM 4.7.0 | pandas 2.2.3 | threads 8 | Colab False
[setup] RAM in use: 0.2 GB


## Section 1 · Load and check the features

Features keep their step-2 numbers (#1–28). The cell stops if:
- a file's columns differ from the list below, or are in a different order;
- a file's rows don't line up with `exposures.parquet`;
- `sid`, `wid`, `label` or `split` is among the features;
- random and cart popularity don't give step 3's validation scores (0.0815 and 0.3342). This checks labels, splits and the metric together.

In [2]:
ex = pd.read_parquet(f"{OUT}/exposures.parquet", columns=["sid", "wid", "label", "split"])
SID, LAB = ex.sid.to_numpy(), ex.label.to_numpy()
IN_SPLIT = {name: (ex.split == code).to_numpy() for name, code in SPLITS.items()}
va = IN_SPLIT["valid"]

FILES = {
    "section1_user_product": ["item_in_history", "item_action_7d_count", "days_since_item_action_7d",            # 1-6
                              "item_cart_90d_count", "item_order_1y_count", "days_since_item_order_1y"],
    "section2_affinity":     ["shop_share_7d", "brand_share_7d", "cate4_share_7d",                               # 7-9
                              "shop_share_cart_90d", "brand_share_cart_90d", "cate4_share_cart_90d",             # 10-12
                              "shop_share_order_1y", "brand_share_order_1y", "cate4_share_order_1y",             # 13-15
                              "qmatch_cate4_share_7d", "qmatch_shop_share_7d"],                                  # 16-17
    "section3_popularity":   ["pop_click_7d", "pop_click_7d_pct", "pop_cart_90d", "pop_cart_90d_pct",            # 18-21
                              "pop_order_1y", "pop_order_1y_pct", "is_cold"],                                    # 22-24
    "section5_text":         ["title_overlap_count", "title_overlap_share", "title_overlap_rank", "title_len"],  # 25-28
}   # section6_label_rates (#29-36) left out: same-day outcomes
FEATURES = [c for names in FILES.values() for c in names]
FILE_OF  = {c: file for file, names in FILES.items() for c in names}

assert len(set(FEATURES)) == 28 and not {"sid", "wid", "label", "split"} & set(FEATURES)
assert (np.diff(SID) >= 0).all()                                # rows are grouped by search (LightGBM needs this)
for file, names in FILES.items():
    path = f"{FEAT}/{file}.parquet"
    assert pq.read_schema(path).names == ["sid", "wid"] + names, file
    ids = pq.read_table(path, columns=["sid", "wid"])
    assert np.array_equal(ids["sid"].to_numpy(), SID) and np.array_equal(ids["wid"].to_numpy(), ex.wid.to_numpy()), file
del ids; gc.collect()

STEP3 = pd.DataFrame({"train": [0.0809, 0.3326, 0.3158, 0.3261, 0.3256],     # step 3, Section 6
                      "valid": [0.0815, 0.3342, 0.3159, 0.3271, 0.3260],
                      "test":  [0.0804, 0.3329, 0.3168, 0.3290, 0.3267]},
                     index=["random", "popularity", "mf", "ncf", "nmf"])

random_score = np.random.default_rng(SEED).random(len(ex))
pop_score = pq.read_table(f"{FEAT}/section3_popularity.parquet", columns=["pop_cart_90d"])["pop_cart_90d"].to_numpy()   # step 3's popularity
check = {"random": ndcg_at_k(SID[va], LAB[va], random_score[va]), "popularity": ndcg_at_k(SID[va], LAB[va], pop_score[va])}
for name, score in check.items():
    assert round(score, 4) == STEP3.valid[name], f"{name}: {score:.4f} here vs {STEP3.valid[name]} in step 3"
print(f"{len(FEATURES)} features checked across {len(FILES)} files | rows {len(ex):,} | "
      + " | ".join(f"{k} {ex.sid[m].nunique():,} searches" for k, m in IN_SPLIT.items()))
print("step 3 repeated on validation:", " | ".join(f"{k} {v:.4f}" for k, v in check.items()))
del random_score; gc.collect(); ram("section 1")

28 features checked across 4 files | rows 15,510,012 | train 120,209 searches | valid 25,759 searches | test 25,760 searches
step 3 repeated on validation: random 0.0815 | popularity 0.3342
[section 1] RAM in use: 0.7 GB


## Section 2 · Tuning

- **Model:** LambdaRank. It learns from pairs of products in the same list, each pair weighted by how much swapping them would change NDCG. So it learns from products that were shown and ignored, which step 3's CF models never saw.
- Missing values stay missing. LightGBM learns which side of a split they go to, so "no history" stays apart from a true 0.
- **Search:** start from `START` (a standard LambdaRank setup) and change one setting at a time. Each run trains on 25% of training searches, is scored on all of validation, and stops after 100 rounds without a gain.
- A change is kept only if validation rises by at least `MIN_GAIN` (0.0005). Later settings are tried on top of the kept ones.

| Setting tried | Valid NDCG@10 | Kept |
|---|---|---|
| `START` | 0.4041 | — |
| `max_bin` 255 | 0.4042 | no |
| `lambdarank_truncation_level` 10 / 30 | 0.4031 / 0.4040 | no |
| `num_leaves` 63 / 127 | 0.4032 / 0.4012 | no |
| **`min_data_in_leaf` 500** | **0.4050** | **yes** |
| `feature_fraction` 0.6 | 0.4046 | no |
| `lambda_l2` 10 | 0.4043 | no |
| `rank_xendcg` objective | 0.3988 | no |
| `learning_rate` 0.03 | 0.4042 | no |

- Only `min_data_in_leaf` 500 clears `MIN_GAIN` (+0.0009). The tuned settings are `START` plus `min_data_in_leaf` 500.
- Bigger trees do worse: 63 leaves −0.0009, 127 leaves −0.0029.
- `rank_xendcg` is 0.0063 below LambdaRank (0.3988 vs 0.4050).

In [3]:
START = dict(objective="lambdarank", metric="ndcg", eval_at=[10], label_gain=[0, 1, 3, 7],   # label_gain = GAIN
             learning_rate=0.05, num_leaves=31, min_data_in_leaf=100, lambda_l2=1.0,
             feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
             max_bin=63, lambdarank_truncation_level=15,
             seed=SEED, deterministic=True, force_col_wise=True, num_threads=os.cpu_count(), verbosity=-1)
ROUNDS, PATIENCE = 3000, 100               # max rounds; stop after this many rounds without a validation gain
MIN_GAIN = 0.0005                          # smallest validation gain worth keeping
RUNS = f"{LGB}/runs.csv"                   # one row per finished run

def load_matrix(rows, cols=FEATURES):
    """float32 matrix of the chosen rows and features, filled one column at a time to limit RAM."""
    X = np.empty((int(rows.sum()), len(cols)), dtype=np.float32)
    for j, c in enumerate(cols):
        X[:, j] = pq.read_table(f"{FEAT}/{FILE_OF[c]}.parquet", columns=[c])[c].to_numpy().astype(np.float32)[rows]
    return X

def group_sizes(rows):
    """Products per search, in row order. Tells LightGBM where each list starts."""
    return np.unique(SID[rows], return_counts=True)[1]

def fit(rows, params, cols=FEATURES, log_every=0):
    """Train on `rows`, early-stopping on validation NDCG@10. The booster predicts with its best round."""
    X = load_matrix(rows, cols)
    train_set = lgb.Dataset(X, label=LAB[rows], group=group_sizes(rows), feature_name=cols, params=params).construct()
    del X; gc.collect()
    valid_set = train_set.create_valid(X_valid[:, [FEATURES.index(c) for c in cols]], label=LAB[va], group=group_sizes(va))
    callbacks = [lgb.early_stopping(PATIENCE, verbose=False)] + ([lgb.log_evaluation(log_every)] if log_every else [])
    return lgb.train(params, train_set, num_boost_round=ROUNDS, valid_sets=[valid_set], valid_names=["valid"], callbacks=callbacks)

def run(name, rows, params, cols=FEATURES):
    """One logged run. Runs already in runs.csv are read back, so a restart picks up where it stopped."""
    done = pd.read_csv(RUNS, index_col="run") if os.path.exists(RUNS) else None
    if done is not None and name in done.index:
        row = done.loc[name]
    else:
        t = time.time()
        booster = fit(rows, params, cols)
        score = booster.predict(X_valid[:, [FEATURES.index(c) for c in cols]])
        row = pd.Series({"valid": ndcg_at_k(SID[va], LAB[va], score), "rounds": booster.best_iteration,
                         "features": len(cols), "minutes": (time.time() - t) / 60}, name=name)
        pd.DataFrame([row]).rename_axis("run").to_csv(RUNS, mode="a", header=done is None)
        del booster, score; gc.collect()
    print(f"{name:>34s}   valid {row.valid:.4f} | {int(row.rounds):>4d} rounds | {row.minutes:4.1f} min")
    return float(row.valid)

X_valid = load_matrix(va)
train_sids = np.unique(SID[IN_SPLIT["train"]])
tune_sids = np.random.default_rng(SEED).choice(train_sids, size=round(TUNE_SHARE * len(train_sids)), replace=False)
TUNE_ROWS = np.isin(SID, tune_sids)
print(f"tuning sample: {len(tune_sids):,} searches, {TUNE_ROWS.sum():,} rows | validation: {va.sum():,} rows")

SWEEP = [("max_bin", [255]),               # tried in this order, each on top of what was kept before
         ("lambdarank_truncation_level", [10, 30]),
         ("num_leaves", [63, 127]),
         ("min_data_in_leaf", [500]),
         ("feature_fraction", [0.6]),
         ("lambda_l2", [10.0]),
         ("objective", ["rank_xendcg"]),
         ("learning_rate", [0.03])]

TUNED, best = dict(START), run("start", TUNE_ROWS, START)
for key, values in SWEEP:
    scores = {v: run(f"{key}={v}", TUNE_ROWS, {**TUNED, key: v}) for v in values}
    v = max(scores, key=scores.get)
    if scores[v] >= best + MIN_GAIN:
        TUNED[key], best = v, scores[v]
print("\nkept:", {k: v for k, v in TUNED.items() if v != START[k]} or "start settings", f"| valid {best:.4f}")
ram("section 2")

tuning sample: 30,052 searches, 2,686,274 rows | validation: 2,303,929 rows
                             start   valid 0.4041 |  393 rounds |  1.8 min
                       max_bin=255   valid 0.4042 |  411 rounds |  1.9 min
    lambdarank_truncation_level=10   valid 0.4031 |  283 rounds |  1.9 min
    lambdarank_truncation_level=30   valid 0.4040 |  279 rounds |  1.9 min
                     num_leaves=63   valid 0.4032 |  212 rounds |  1.5 min
                    num_leaves=127   valid 0.4012 |  203 rounds |  1.6 min
              min_data_in_leaf=500   valid 0.4050 |  342 rounds |  1.9 min
              feature_fraction=0.6   valid 0.4046 |  422 rounds |  2.1 min
                    lambda_l2=10.0   valid 0.4043 |  452 rounds |  2.2 min
             objective=rank_xendcg   valid 0.3988 |  470 rounds |  1.7 min
                learning_rate=0.03   valid 0.4042 |  328 rounds |  1.9 min

kept: {'min_data_in_leaf': 500} | valid 0.4050
[section 2] RAM in use: 0.9 GB


## Section 3 · Ablations by step-2 section

Each run drops one step-2 section and retrains the tuned model on the same 25% sample. The drop in validation NDCG@10 is what that section adds on top of the others.

| Removed | Valid NDCG@10 | Change |
|---|---|---|
| nothing (all 28) | 0.4050 | — |
| **Section 3 (popularity)** | 0.2244 | **−0.1806** |
| Section 1 (user × product) | 0.3791 | −0.0259 |
| Section 5 (text) | 0.3976 | −0.0075 |
| Section 2 (affinity) | 0.3986 | −0.0065 |

- Popularity matters most. Without it the model falls to 0.2244, well below popularity alone (0.3342), so the other sections only work on top of it.
- User × product is next (−0.0259), even though it scored only 0.174 on its own in step 2.
- Text and affinity add less. Title match was barely above random on its own in step 2 (0.092), but it still adds 0.0075 here.

In [4]:
DROPS = {"without section 1 (user x product)": FILES["section1_user_product"],
         "without section 2 (affinity)":      FILES["section2_affinity"],
         "without section 3 (popularity)":    FILES["section3_popularity"],
         "without section 5 (text)":          FILES["section5_text"]}

ablation = {"all 28": best}
for name, drop in DROPS.items():
    ablation[name] = run(name, TUNE_ROWS, TUNED, [c for c in FEATURES if c not in drop])

ablation = pd.DataFrame({"valid": ablation})
ablation["change"] = ablation.valid - best
print("\n", ablation.round(4).to_string())
ram("section 3")

without section 1 (user x product)   valid 0.3791 |  562 rounds |  2.4 min
      without section 2 (affinity)   valid 0.3986 |  377 rounds |  1.7 min
    without section 3 (popularity)   valid 0.2244 |  377 rounds |  1.8 min
          without section 5 (text)   valid 0.3976 |  256 rounds |  1.5 min

                                      valid  change
all 28                              0.4050  0.0000
without section 1 (user x product)  0.3791 -0.0259
without section 2 (affinity)        0.3986 -0.0065
without section 3 (popularity)      0.2244 -0.1806
without section 5 (text)            0.3976 -0.0075
[section 3] RAM in use: 0.9 GB


## Section 4 · Final models

Both models train on all training searches with all 28 features:
- `start`: the `START` settings, to show what tuning adds.
- `tuned`: the settings kept in Section 2. This is the final model. If tuning kept nothing, `tuned` is the same as `start` and isn't trained twice.

A saved model is reloaded instead of retrained.

| Model | Best round | Valid NDCG@10 (LightGBM's own) |
|---|---|---|
| `start` | 499 | 0.4060 |
| **`tuned`** | 736 | 0.4065 |

Section 6 rescores both with `ndcg_at_k`, the metric used for every other model.

In [5]:
def final(name, params, cols=FEATURES):
    """Train on all training searches (or reload the saved model), then save one score per shown product."""
    path = f"{LGB}/model_{name}.txt"
    if os.path.exists(path):
        booster = lgb.Booster(model_file=path)
        print(f"{name}: reloaded from {path} ({booster.current_iteration()} rounds)")
    else:
        t = time.time()
        booster = fit(IN_SPLIT["train"], params, cols, log_every=100)
        booster.save_model(path)
        print(f"{name}: best round {booster.best_iteration} | LightGBM's own valid NDCG@10 "
              f"{booster.best_score['valid']['ndcg@10']:.4f} | {(time.time() - t) / 60:.1f} min")
    score = np.empty(len(ex), dtype=np.float32)
    for a in range(0, len(ex), 3_000_000):                       # 3M rows at a time to limit RAM
        rows = np.zeros(len(ex), dtype=bool)
        rows[a:a + 3_000_000] = True
        score[rows] = booster.predict(load_matrix(rows, cols))
    ex[["sid", "wid"]].assign(score=score).to_parquet(f"{LGB}/score_{name}.parquet", index=False, compression="zstd")
    return booster, score

models, scores = {}, {}
for name, params in [("start", START), ("tuned", TUNED)]:
    if name == "tuned" and TUNED == START:                       # tuning kept nothing: same model
        models[name], scores[name] = models["start"], scores["start"]
        continue
    models[name], scores[name] = final(name, params)
    gc.collect(); ram(name)

del X_valid; _ = gc.collect()

start: reloaded from ./lightgbm/model_start.txt (499 rounds)


[start] RAM in use: 0.8 GB
tuned: reloaded from ./lightgbm/model_tuned.txt (736 rounds)


[tuned] RAM in use: 0.5 GB


## Section 5 · Feature importance

Gain = total improvement in the objective from every split on a feature. Related features share credit, so Section 3's ablations are the better guide to what a section adds.

| Section | Share of gain (`tuned`) |
|---|---|
| 3 popularity | 66.5% |
| 1 user × product | 23.7% |
| 2 affinity | 5.1% |
| 5 text | 4.7% |

- Three features carry 78% of the gain: `pop_cart_90d_pct` (43.5%), `item_in_history` (20.0%) and `pop_click_7d_pct` (14.4%).
- The `_pct` features (popularity percentile within the list) get far more gain than the raw counts (`pop_cart_90d_pct` 43.5% vs `pop_cart_90d` 2.3%). How a product compares with the rest of its list matters more than its raw popularity.

In [6]:
SECTION_NAME = {"section1_user_product": "1 user x product", "section2_affinity": "2 affinity",
                "section3_popularity": "3 popularity", "section5_text": "5 text"}
gain = models["tuned"].feature_importance("gain")
importance = pd.DataFrame({"number": range(1, len(FEATURES) + 1), "feature": FEATURES,
                           "section": [SECTION_NAME[FILE_OF[c]] for c in FEATURES],
                           "gain_share": gain / gain.sum(),
                           "splits": models["tuned"].feature_importance("split")}).sort_values("gain_share", ascending=False)
importance.to_csv(f"{LGB}/importance.csv", index=False)

print(importance.head(12).round(4).to_string(index=False))
print("\nshare of gain by section:")
print(importance.groupby("section").gain_share.sum().sort_values(ascending=False).round(4).to_string())

 number                   feature          section  gain_share  splits
     21          pop_cart_90d_pct     3 popularity      0.4355    1877
      1           item_in_history 1 user x product      0.2003     191
     19          pop_click_7d_pct     3 popularity      0.1436    2018
     23          pop_order_1y_pct     3 popularity      0.0359    1734
     20              pop_cart_90d     3 popularity      0.0231    1567
      2      item_action_7d_count 1 user x product      0.0183     263
     18              pop_click_7d     3 popularity      0.0170    1779
      7             shop_share_7d       2 affinity      0.0153     728
      3 days_since_item_action_7d 1 user x product      0.0144     736
     28                 title_len           5 text      0.0143    1984
     26       title_overlap_share           5 text      0.0131     896
     27        title_overlap_rank           5 text      0.0130     752

share of gain by section:
section
3 popularity        0.6654
1 user x produc

## Section 6 · Train and validation results

All scores use `ndcg_at_k`, same as step 3. Gaps are paired: both models scored on the same validation searches.

| Model | Train | Valid |
|---|---|---|
| random | 0.0809 | 0.0815 |
| MF | 0.3158 | 0.3159 |
| NMF | 0.3256 | 0.3260 |
| NCF | 0.3261 | 0.3271 |
| popularity (carts, 90 days) | 0.3326 | 0.3342 |
| LightGBM, start | 0.4159 | 0.4061 |
| **LightGBM, tuned** | 0.4208 | **0.4065** |

| Comparison (validation) | Difference | 95% interval | p-value |
|---|---|---|---|
| tuned − popularity | +0.0723 | +0.0692 to +0.0755 | < 0.0001 |
| tuned − start | +0.0004 | −0.0005 to +0.0013 | 0.38 |

- LightGBM beats popularity, the best baseline, by 0.0723 (22% higher).
- Tuning adds only 0.0004 and the interval includes 0, so `start` and `tuned` are about the same.
- Train is 0.014 above validation for `tuned` and 0.010 for `start` (which stops earlier, at 499 rounds vs 736). Mild overfitting.
- Validation picked the settings, so these scores may run slightly high. Section 7 has the test scores.

In [7]:
NAMES = {"start": "lightgbm, start", "tuned": "lightgbm, tuned"}
if TUNED == START:                          # tuning kept nothing: one model, one row
    NAMES = {"start": "lightgbm, 28 features"}

def evaluate(score, splits=("train", "valid")):
    return {split: ndcg_at_k(SID[IN_SPLIT[split]], LAB[IN_SPLIT[split]], score[IN_SPLIT[split]]) for split in splits}

table = STEP3[["train", "valid"]].copy()
for name, label in NAMES.items():
    table.loc[label] = evaluate(scores[name])
table["gap to train"] = table.train - table.valid
table.to_csv(f"{LGB}/results.csv")
print(table.round(4).to_string())

COMPARE = [("tuned", "popularity")] + ([("tuned", "start")] if "tuned" in NAMES else [])
per_search = {name: ndcg_lists(SID[va], LAB[va], s[va]) for name, s in {**scores, "popularity": pop_score}.items()}
paired = pd.DataFrame({f"{a} - {b}": paired_test(per_search[a], per_search[b]) for a, b in COMPARE}).T
print("\npaired gaps on validation:")
print(paired.round(4).to_string())

                  train   valid  gap to train
random           0.0809  0.0815       -0.0006
popularity       0.3326  0.3342       -0.0016
mf               0.3158  0.3159       -0.0001
ncf              0.3261  0.3271       -0.0010
nmf              0.3256  0.3260       -0.0004
lightgbm, start  0.4159  0.4061        0.0098
lightgbm, tuned  0.4208  0.4065        0.0142



paired gaps on validation:
                    difference  95% low  95% high  p-value  share better  share worse
tuned - popularity      0.0723   0.0692    0.0755   0.0000        0.3577       0.2003
tuned - start           0.0004  -0.0005    0.0013   0.3779        0.1056       0.1013


## Section 7 · Test results

Each model's saved scores are scored once on test (25,760 searches). Nothing is changed after this. Paired tests compare two models on the same test searches.

| Model | Train | Valid | Test |
|---|---|---|---|
| random | 0.0809 | 0.0815 | 0.0804 |
| MF | 0.3158 | 0.3159 | 0.3168 |
| NMF | 0.3256 | 0.3260 | 0.3267 |
| NCF | 0.3261 | 0.3271 | 0.3290 |
| popularity (carts, 90 days) | 0.3326 | 0.3342 | 0.3329 |
| LightGBM, start | 0.4159 | 0.4061 | 0.4062 |
| **LightGBM, tuned** | 0.4208 | 0.4065 | **0.4063** |

| Comparison (test) | Difference | 95% interval | p-value | Better / worse searches |
|---|---|---|---|---|
| tuned − popularity | +0.0734 | +0.0702 to +0.0765 | < 0.0001 | 36% / 20% |
| tuned − start | +0.0001 | −0.0008 to +0.0010 | 0.87 | 11% / 10% |

- The final model scores 0.4063 on test, 0.0734 above popularity.
- Test is within 0.0002 of validation, so picking settings on validation didn't inflate the score.
- Tuning makes no difference on test (+0.0001, p = 0.87).
- LightGBM still does worse than popularity on 20% of test searches.

In [8]:
if RUN_TEST:
    m = IN_SPLIT["test"]
    per_search_test = {name: ndcg_lists(SID[m], LAB[m], s[m]) for name, s in {**scores, "popularity": pop_score}.items()}
    test = pd.Series({name: per_search_test[name].mean() for name in NAMES}, name="test")
    test.to_csv(f"{LGB}/test.csv")

    table["test"] = pd.concat([STEP3.test, test.rename(NAMES)])
    print(f"test searches: {len(per_search_test['popularity']):,}")
    print(table[["train", "valid", "test"]].round(4).to_string())

    paired_test_set = pd.DataFrame({f"{a} - {b}": paired_test(per_search_test[a], per_search_test[b]) for a, b in COMPARE}).T
    paired_test_set.to_csv(f"{LGB}/paired_tests.csv")
    print("\npaired gaps on test:")
    print(paired_test_set.round(4).to_string())
else:
    print("test not scored: RUN_TEST is False")

test searches: 25,760
                  train   valid    test
random           0.0809  0.0815  0.0804
popularity       0.3326  0.3342  0.3329
mf               0.3158  0.3159  0.3168
ncf              0.3261  0.3271  0.3290
nmf              0.3256  0.3260  0.3267
lightgbm, start  0.4159  0.4061  0.4062
lightgbm, tuned  0.4208  0.4065  0.4063

paired gaps on test:
                    difference  95% low  95% high  p-value  share better  share worse
tuned - popularity      0.0734   0.0702    0.0765    0.000        0.3610       0.2015
tuned - start           0.0001  -0.0008    0.0010    0.867        0.1073       0.1023


## Section 8 · Segments

Validation only. Validation NDCG@10 per group of searches, for popularity and both LightGBM models. The groups are the same as in the KGIN and LSTM notebooks:
- **cold products:** how many of the search's interacted products are in no one's history before 17 Oct (`is_cold`);
- **activity:** the user's clicks in the last 7 days (step 2, Section 4);
- **new vs repeat search:** repeat = in the last 7 days the user acted on results of a similar query (word overlap ≥ 0.5), i.e. `qmatch_cate4_share_7d` is set.

| Group | Searches | Popularity | LightGBM tuned | Gain |
|---|---|---|---|---|
| interacted products: none cold | 17,567 | 0.4274 | 0.4982 | +0.0708 |
| some cold | 3,561 | 0.2070 | 0.2744 | +0.0674 |
| all cold | 4,631 | 0.0733 | 0.1594 | +0.0861 |
| clicks in 7 days: 0 | 4,184 | 0.3468 | 0.3886 | +0.0418 |
| 1–20 | 9,648 | 0.3336 | 0.4131 | +0.0795 |
| 21–100 | 8,605 | 0.3321 | 0.4088 | +0.0767 |
| over 100 | 3,322 | 0.3206 | 0.4040 | +0.0833 |
| new search | 22,496 | 0.3386 | 0.3981 | +0.0595 |
| repeat search | 3,263 | 0.3032 | 0.4642 | +0.1610 |

- LightGBM beats popularity in every group.
- When every interacted product is cold, LightGBM more than doubles popularity (0.1594 vs 0.0733). Popularity counts are 0 for these products, so the model ranks them through history, affinity and title match instead. This is still by far the hardest group.
- Users with no clicks in the last 7 days gain least (+0.0418, against +0.077 to +0.083 for the rest), since the 7-day history features are empty for them.
- Repeat searches gain most (+0.1610 vs +0.0595 for new ones): the user's own recent actions point straight at what they want.

In [9]:
m = va                                                        # validation only; test stays untouched
pos = m & (LAB > 0)                                           # interacted rows
is_cold = pq.read_table(f"{FEAT}/section3_popularity.parquet", columns=["is_cold"])["is_cold"].to_numpy() == 1
cold_share = pd.Series(is_cold[pos]).groupby(SID[pos]).mean() # share of the search's interacted products that are cold
cold = pd.Series(np.select([cold_share == 0, cold_share == 1], ["none cold", "all cold"], "some cold"), index=cold_share.index)
ctx = pd.read_parquet(f"{FEAT}/section4_context.parquet", columns=["sid", "clicks_7d"]).set_index("sid")
activity = pd.cut(ctx.clicks_7d, [-1, 0, 20, 100, np.inf], labels=["0", "1-20", "21-100", "over 100"]).astype(str)
qm = pd.read_parquet(f"{FEAT}/section2_affinity.parquet", columns=["sid", "qmatch_cate4_share_7d"])
repeat = qm.qmatch_cate4_share_7d.notna().groupby(qm.sid).any().map({True: "repeat", False: "new"})

GROUPS = {"interacted products": (cold, ["none cold", "some cold", "all cold"]),
          "clicks in the last 7 days": (activity, ["0", "1-20", "21-100", "over 100"]),
          "search": (repeat, ["new", "repeat"])}
seg_scores = {"popularity": pop_score, **{NAMES[n]: scores[n] for n in NAMES}}
rows = []
for title, (group, order) in GROUPS.items():
    g_row = group.reindex(SID).to_numpy()                     # group of every row
    for g in order:
        s = m & (g_row == g)
        rows.append({"grouping": title, "group": g, "searches": len(np.unique(SID[s])),
                     **{n: ndcg_at_k(SID[s], LAB[s], sc[s]) for n, sc in seg_scores.items()}})
segments = pd.DataFrame(rows).set_index(["grouping", "group"])
segments["gain over popularity"] = segments[list(NAMES.values())[-1]] - segments.popularity
segments.to_csv(f"{LGB}/segments.csv")
print(segments.round(4).to_string())
del ctx, qm, is_cold; gc.collect(); ram("section 8")

                                     searches  popularity  lightgbm, start  lightgbm, tuned  gain over popularity
grouping                  group                                                                                  
interacted products       none cold     17567      0.4274           0.4977           0.4982                0.0708
                          some cold      3561      0.2070           0.2737           0.2744                0.0674
                          all cold       4631      0.0733           0.1598           0.1594                0.0861
clicks in the last 7 days 0              4184      0.3468           0.3870           0.3886                0.0418
                          1-20           9648      0.3336           0.4130           0.4131                0.0795
                          21-100         8605      0.3321           0.4086           0.4088                0.0767
                          over 100       3322      0.3206           0.4032           0.4